# Hansen Ch.8 Restricted Estimation — 计算

**Chapter 8 Restricted Estimation**

完整推导与**面向初学者的详细注释**见同目录 `Hansen_Ch08_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook：**Exercise 8.19**（CLS 与有效 MD 实操）。

> **写给只学过李子奈/陈强的同学：** 本章讲**当经济理论给出约束 $R'\beta=c$ 时如何利用它**。两套等价工具：
> - **CLS（约束最小二乘）**：约束下最小化 SSE，Stata `cnsreg`。
> - **MD（最小距离）**：把无约束 $\hat\beta$ 投影到约束集合，权重 $W$。CLS 是 MD 的特例（$W=\hat Q_{XX}$）。
>
> 核心直觉——**施加正确约束让方差下降**（"减去一个半正定项"）：
> $$V_{\tilde\beta,\text{有效}}=V_\beta-V_\beta R(R'V_\beta R)^{-1}R'V_\beta\le V_\beta.$$
> 但约束**为真**才有此好处；约束**错**了会引入偏差（且让 SSE 上升，见 Ex 8.21）。
> 有效 MD 用 $W=V_\beta^{-1}$（方差倒数作权重），同方差下退化为 CLS，异方差下优于 CLS。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

def ols_hc3(y, X):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    u = X * (e / np.clip(1 - h, 1e-12, None))[:, None]
    V = XXinv @ (u.T @ u) @ XXinv
    return beta, e, V, n, k

CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100
s = df[(df.race == 1) & (df.female == 0) & (df.hisp == 1)].copy().reset_index(drop=True)
for code, name in [(1, "m1"), (2, "m2"), (3, "m3"), (4, "wid"), (5, "div"), (6, "sep")]:
    s[name] = (s.marital == code).astype(float)
y = s.lwage.to_numpy()
X = np.column_stack([
    s.education, s.experience, s.exp2, s.m1, s.m2, s.m3, s.wid, s.div, s.sep, np.ones(len(s))
])
names = ["edu", "exp", "exp2", "m1", "m2", "m3", "wid", "div", "sep", "int"]
beta, e, V, n, k = ols_hc3(y, X)
print("n =", n)
print(pd.DataFrame({"beta": beta, "HC3_SE": np.sqrt(np.diag(V))}, index=names))

# CLS: m1=wid, div=sep
Xcls = np.column_stack([
    s.education, s.experience, s.exp2, s.m1 + s.wid, s.m2, s.m3, s.div + s.sep, np.ones(len(s))
])
bcls, _, Vcls, _, _ = ols_hc3(y, Xcls)
print("\nCLS (reparameterized):", bcls)

# EMD
R = np.array([[0, 0, 0, 1, 0, 0, -1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, -1, 0]], float)
bmd = beta - V @ R.T @ np.linalg.inv(R @ V @ R.T) @ (R @ beta)
print("\nEMD:")
print(pd.Series(bmd, index=names))
print("d(exp)/d at 0 and 50:", beta[1], beta[1] + beta[2])


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch08：CLS 公式=重参数化、$s^2_{\mathrm{cls}}$ 在 df=$n-k+q$ 下无偏（约束为真）、有效 MD 方差"减去半正定项"、两样本逆方差加权。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(8)

# Ex 8.5 / 8.9: CLS 公式 (8.8) 与重参数化一致；约束为真时 s²_cls(df=n-k+q) 无偏
n, k, q = 80, 2, 1
R = np.array([[1.0], [0.0]]); c = np.array([3.0]); sigma2 = 4.0
X = np.c_[rng.standard_normal(n), np.ones(n)]
beta = np.array([3.0, 1.0])                                     # 约束为真: β₁=3
s2_nkq = []
for r in range(40000):
    e = rng.standard_normal(n) * np.sqrt(sigma2); Y = X @ beta + e
    bhat = np.linalg.solve(X.T @ X, X.T @ Y); Qinv = np.linalg.inv(X.T @ X)
    bcls = bhat - Qinv @ R @ np.linalg.inv(R.T @ Qinv @ R) @ (R.T @ bhat - c)   # CLS 公式 (8.8)
    bcls2 = np.array([3.0, np.mean(Y - 3.0 * X[:, 0])])        # 重参数化: 固定 β₁=3 后对 X₂ 回归
    assert np.allclose(bcls, bcls2)
    te = Y - X @ bcls
    s2_nkq.append((te @ te) / (n - k + q))                     # df = n - k + q
print(f"[8.5/8.9] CLS公式=重参数化(逐样本一致); E[s²_cls(df=n-k+q)]={np.mean(s2_nkq):.4f} = σ²={sigma2}")

# Ex 8.26: 有效 MD 方差 = V - V·R·(R'VR)^-1·R'·V（减去半正定项 ⇒ 施加正确约束降方差）
Q = np.array([[2.0, 0.3], [0.3, 1.0]])
Om = np.array([[3.0, 0.5], [0.5, 2.0]])
Vb = np.linalg.solve(Q, Om @ np.linalg.inv(Q))                 # Q^-1 Ω Q^-1
R2 = np.array([[1.0], [-1.0]])                                  # 约束 β₁-β₂=0
Vemd = Vb - Vb @ R2 @ np.linalg.inv(R2.T @ Vb @ R2) @ R2.T @ Vb
print(f"[8.26] V - V_emd 特征值={np.round(np.linalg.eigvalsh(Vb - Vemd), 4)} (≥0 ⇒ 方差下降)")

# Ex 8.18: 两独立样本的逆方差加权（信息量加权汇总）
b, N, reps = 1.0, 100, 30000
est = []
for r in range(reps):
    X1 = rng.standard_normal(N); X2 = rng.standard_normal(N)
    e1 = rng.standard_normal(N) * np.sqrt(2.0); e2 = rng.standard_normal(N)
    b1 = np.sum(X1*(X1*b + e1)) / np.sum(X1**2)
    b2 = np.sum(X2*(X2*b + e2)) / np.sum(X2**2)
    v1, v2 = 2.0/np.sum(X1**2), 1.0/np.sum(X2**2)
    est.append((b1/v1 + b2/v2) / (1/v1 + 1/v2))
print(f"[8.18] 两样本加权 var={np.var(est):.6f} ≈ (信息量倒数和)⁻¹={1/(N/2 + N/1):.6f}")